# Image to video with gpt-4o and Sora

Sora is an AI model from OpenAI that can create realistic and imaginative video scenes from text instructions. The model is capable of generating a wide range of video content, including realistic scenes, animations, and special effects. Several video resolutions and durations are supported.

https://learn.microsoft.com/en-us/azure/ai-services/openai/concepts/video-generation

In [ ]:
import base64
import cv2
import datetime
import numpy as np
import openai
import os
import requests
import sys
import time

from dotenv import load_dotenv
from io import BytesIO
from IPython.display import Image, Video, FileLink, Audio
from mimetypes import guess_type
from openai import AzureOpenAI

In [ ]:
sys.version

In [ ]:
print(f"Today is {datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

## Settings

In [ ]:
load_dotenv("azure.env")

endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
api_key = os.getenv('AZURE_OPENAI_API_KEY')

api_version = "2025-04-01-preview"
model = "gpt-4.1"
sora_model = "sora"

In [ ]:
IMAGES_DIR = "images"

In [ ]:
OUTPUT_DIR = "videos"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Helper Function to convert image to data URL

In [ ]:
def local_image_to_data_url(image_path):
    """
     
 Converts a local image file to a data URL.

Parameters:
    image_path (str): Path to the local image file.
 Returns:
    str: A data URL containing the base64-encoded image data.
    """
    mime_type, _ = guess_type(image_path)

    if mime_type is None:
        mime_type = "application/octet-stream"

    with open(image_path, "rb") as image_file:
        base64_encoded_data = base64.b64encode(
            image_file.read()).decode("utf-8")

    return f"data:{mime_type};base64,{base64_encoded_data}"

## Helper function to analyze an image

In [ ]:
def describe_image_with_gpt41(image_file, prompt):
    """
    Uses GPT-4.1 to analyze an image.

    Parameters:
    image_file (str): Path to the image file to be analyzed.
    prompt (str): Text prompt describing what to analyze in the image.

    Returns:
    str: The description generated by GPT-4o based on the image and prompt.

    The function initializes an AzureOpenAI client, sends a request to the GPT-4o model with the image and prompt, 
    and returns the generated description. The image is converted to a data URL before being sent to the model.
    """
    client = AzureOpenAI(
        api_key=api_key,
        api_version=api_version,
        base_url=f"{endpoint}/openai/deployments/{model}",
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role":
                "system",
                "content":
                "You are an AI helpful assistant to analyse an image.",
            },
            {
                "role":
                "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": local_image_to_data_url(image_file)
                        },
                    },
                ],
            },
        ],
        max_tokens=2000,
        temperature=0.7,
    )

    return response.choices[0].message.content

## Helper function to generate video

In [ ]:
def sora(prompt, width=480, height=480, n_seconds=5):
    """
    Generates a video based on the given prompt using the SORA model.

    Parameters:
    prompt (str): The text prompt to generate the video.
    width (int): The width of the video. Supported values are 480, 854, 720, 1080, and 1920.
    height (int): The height of the video. Supported values are 480, 854, 720, 1080, and 1920.
    n_seconds (int): The duration of the video in seconds. Must be between 1 and 20 seconds.
    n_variants (int): The number of video variants to generate.
    
    Returns:
    str: The filename of the generated video.

    Raises:
    Exception: If the video generation job fails or no generations are found.
    """
    start = time.time()

    api_version = 'preview'
    headers = {"api-key": api_key, "Content-Type": "application/json"}

    idx = datetime.datetime.today().strftime('%d%b%Y_%H%M%S')
    output_filename = os.path.join(OUTPUT_DIR, f"image_to_video_sora_{idx}_{image_file.split('/')[-1].split('.')[0]}.mp4")

    # 1. Create a video generation job
    create_url = f"{endpoint}/openai/v1/video/generations/jobs?api-version={api_version}"
    body = {
        "prompt": prompt,
        "width": width,  # 480x480, 480x854, 854x480, 720x720, 720x1280, 1280x720, 1080x1080, 1080x1920, 1920x1080.
        "height": height,  # 480x480, 480x854, 854x480, 720x720, 720x1280, 1280x720, 1080x1080, 1080x1920, 1920x1080.
        "n_seconds": n_seconds,  # between 1 and 20 seconds
        "model": sora_model,  # SORA model
    }
    response = requests.post(create_url, headers=headers, json=body)
    response.raise_for_status()

    now = datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')
    print(f"{now} Full response JSON:", response.json())
    print()

    job_id = response.json()["id"]
    now = datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')
    print(f"{now} Job created: {job_id}")

    # 2. Poll for job status
    status_url = f"{endpoint}/openai/v1/video/generations/jobs/{job_id}?api-version={api_version}"
    status = None

    while status not in ("succeeded", "failed", "cancelled"):
        time.sleep(5)  # Wait before polling again
        status_response = requests.get(status_url, headers=headers).json()
        status = status_response.get("status")
        now = datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')
        print(f"{now} Job status: {status}")

    # 3. Retrieve generated video
    if status == "succeeded":
        generations = status_response.get("generations", [])
        
        if generations:
            now = datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')
            print(f"\n{now} ✅ Done. Video generation succeeded.")
            generation_id = generations[0].get("id")
            video_url = f"{endpoint}/openai/v1/video/generations/{generation_id}/content/video?api-version={api_version}"
            video_response = requests.get(video_url, headers=headers)

            if video_response.ok:
                # Downloading the video
                print("\nDownloading the video...")
                with open(output_filename, "wb") as file:
                    file.write(video_response.content)
                    print(f"SORA Generated video saved: '{output_filename}'")

                elapsed = time.time() - start
                minutes, seconds = divmod(elapsed, 60)
                print(
                    f"Done in {minutes:.0f} minutes and {seconds:.0f} seconds")

                return output_filename
        else:
            raise Exception("Error. No generations found in job result.")
    else:
        raise Exception(f"Error. Job did not succeed. Status: {status}")

## Example 1

In [ ]:
image_file = os.path.join(IMAGES_DIR, "ahhh.jpeg")

Image(image_file, width=640)

In [ ]:
sora_prompt = """
You are a senior art-director creating marketing assets, like videos, for a premium sleep-technology brand.
You are tasked with creating a 7-second video that showcases the comfort and innovation of a new connected mattress.
You will use the SORA model to generate this video based on the following detailed prompt:
- Subject: young black female model lying on a premium mattress, demonstrating its comfort and smart features.
- Action: model starts supine, slowly rolls to side, settles with a smile; subtle bluish vapor ripple animates under her torso to show active cooling; letters compress slightly under weight.
- Setting: modern, minimalistic bedroom with soft ambient lighting; focus on the mattress and model.
- Style: up-market lifestyle ad, emphasizing comfort, innovation, and relaxation.
- Visual tone: cozy, elegant, premium, tech-savvy.   
- Design language: oversized pillowy 3-D letters that spell an exclamation (“AHHH”, “WOAH”, “OOOH”, “YESS”) acting as a mattress; soft microfiber texture; subtle shadows; matte finish.   
- Palette: deep gradient sky-tones behind the scene (cool teal, rich emerald, muted plum, warm caramel); foreground lettering in snow-white with faint icy sparkle.  
- Lighting: studio, soft-box key light, gentle rim light; no harsh contrast.   
- Backdrop overlay: constellation of evenly spaced micro-dots in a radial arc, implying airflow & smart-tech.   
- Talent: mid-30s woman, natural curls, wearing simple sleepwear; expression relaxed and content.   
- Camera: locked-off, eye-level, medium-wide.   Everything should feel like an up-market lifestyle ad for a connected mattress.   
"""
print(sora_prompt)

In [ ]:
sora_prompt = """
Create a scene: a young black woman that is wearing pajamas and sleeping on her side, on a mattress in the shape of the word AHHH, 
her weight compresses the letters very lightly. She start by being face up, then slowly rolls to her left to face de camera. 
The visual style should mimic Photorealistic, emphasizing lifelike details and natural textures. The camera movement should be a Tracking Shot, 
smoothly following the action. The scene takes place during Sunset, with lighting suited to that time. The atmosphere features Rainy conditions, 
adding depth to the environment. The overall mood should feel inspirational, evoking motivation and empowerment. 
"""

In [21]:
generated_video_file = sora(sora_prompt, width=1280, height=720, n_seconds=7)

10-Jun-2025 14:43:10 Full response JSON: {'object': 'video.generation.job', 'id': 'task_01jxdjehpgeh6vs7ttax4qtaah', 'status': 'queued', 'created_at': 1749580990, 'finished_at': None, 'expires_at': None, 'generations': [], 'prompt': 'Create a scene: a young black woman that is wearing pajamas and sleeping on her side, on a mattress in she shape of the word AHHH, her weight compresses the letters very lightly. She start by being face up, then slowly rolls to her left to face de camera. The visual style should mimic Photorealistic, emphasizing lifelike details and natural textures. The camera movement should be a Tracking Shot, smoothly following the action. The scene takes place during Sunset, with lighting suited to that time. The atmosphere features Rainy conditions, adding depth to the environment. The overall mood should feel inspirational, evoking motivation and empowerment. The scene is accompanied by gentle piano music, enhancing the atmospheric immersion.', 'model': 'sora', 'n_v

In [22]:
Video(generated_video_file, width=1024)

In [ ]:
video_link = FileLink(path=generated_video_file)
video_link

In [ ]:
# Mixing the initial image on the left and the SORA video on the right
output_path = os.path.join(OUTPUT_DIR, "image_sora_car.mp4")

left_image = cv2.imread(image_file)
video = cv2.VideoCapture(generated_video_file)

fps = video.get(cv2.CAP_PROP_FPS)
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
left_image_resized = cv2.resize(
    left_image,
    (int(height * left_image.shape[1] / left_image.shape[0]), height))
combined_width = left_image_resized.shape[1] + width
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (combined_width, height))

while True:
    ret, frame = video.read()
    if not ret:
        break
    combined_frame = np.hstack((left_image_resized, frame))
    out.write(combined_frame)

video.release()
out.release()

video_link = FileLink(path=output_path)
video_link

## Example 2

In [ ]:
image_file = os.path.join(IMAGES_DIR, "food.jpg")

Image(image_file, width=640)

In [ ]:
description = describe_image_with_gpt41(image_file, "Describe the following image in detail, including the main objects, their colors, positions, motions and any notable features")

print(description)

In [ ]:
sora_prompt = f"Based on the following descriptions of an image:\n {description} \nPlease create a coherent generative AI prompt that can be used to generate a video."

In [ ]:
generated_video_file = sora(sora_prompt, width=1280, height=720, n_seconds=5)

In [ ]:
Video(generated_video_file, width=1024)

In [ ]:
video_link = FileLink(path=generated_video_file)
video_link

In [ ]:
# Mixing the initial image on the left and the SORA video on the right
output_path = os.path.join(OUTPUT_DIR, "image_sora_food.mp4")

left_image = cv2.imread(image_file)
video = cv2.VideoCapture(generated_video_file)

fps = video.get(cv2.CAP_PROP_FPS)
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
left_image_resized = cv2.resize(
    left_image,
    (int(height * left_image.shape[1] / left_image.shape[0]), height))
combined_width = left_image_resized.shape[1] + width
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (combined_width, height))

while True:
    ret, frame = video.read()
    if not ret:
        break
    combined_frame = np.hstack((left_image_resized, frame))
    out.write(combined_frame)

video.release()
out.release()

video_link = FileLink(path=output_path)
video_link

## Example 3

In [ ]:
image_file = os.path.join(IMAGES_DIR, "cabourg.jpg")

Image(image_file, width=860)

In [ ]:
description = describe_image_with_gpt41(image_file, "Describe the following image in detail, including the main objects, their colors, positions, motions and any notable features")

print(description)

In [ ]:
sora_prompt = f"Based on the following descriptions of an image:\n {description} \nPlease create a coherent generative AI prompt that can be used to generate a video."

print(sora_prompt)

In [ ]:
generated_video_file = sora(sora_prompt, width=1280, height=720, n_seconds=7)

In [ ]:
Video(generated_video_file, width=1024)

In [ ]:
video_link = FileLink(path=generated_video_file)
video_link

In [ ]:
# Mixing the initial image on the left and the SORA video on the right
output_path = os.path.join(OUTPUT_DIR, "image_sora_store.mp4")

left_image = cv2.imread(image_file)
video = cv2.VideoCapture(generated_video_file)

fps = video.get(cv2.CAP_PROP_FPS)
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
left_image_resized = cv2.resize(
    left_image,
    (int(height * left_image.shape[1] / left_image.shape[0]), height))
combined_width = left_image_resized.shape[1] + width
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (combined_width, height))

while True:
    ret, frame = video.read()
    if not ret:
        break
    combined_frame = np.hstack((left_image_resized, frame))
    out.write(combined_frame)

video.release()
out.release()

video_link = FileLink(path=output_path)
video_link

## Generated videos

In [ ]:
!ls -lh videos/image_sora*.mp4 